# Iniciación en análisis de datos con Python

**Mayo 2026 · Bloque I**

## Objetivos
- Configurar el entorno de trabajo en Jupyter/VS Code
- Cargar, explorar y limpiar datos con pandas
- Construir un análisis descriptivo reproducible

## Preparación
Ejecuta la primera celda para cargar librerías. Si falta alguna librería, instálala desde el entorno con `pip install -r requirements.txt`.

## Carga de datos y primera inspección

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

DATA_DIR = Path("../data")
pd.set_option("display.max_columns", 50)

df = pd.read_csv(DATA_DIR / "ventas_mayo_2026.csv", parse_dates=["fecha"])
display(df.head())
df.info()

## Limpieza básica y tipos de datos

In [ ]:
df_clean = df.copy()

# Verificar qué columnas existen
print("Columnas disponibles:", df_clean.columns.tolist())

# Rellenar valores nulos en columnas categóricas
for col in ["categoria", "region", "canal"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("desconocido")

# Rellenar valores nulos en columnas numéricas
for col in ["importe", "unidades", "precio_unitario"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Crear variable derivada: precio con descuento
if "precio_unitario" in df_clean.columns and "descuento" in df_clean.columns:
    df_clean["precio_final"] = df_clean["precio_unitario"] * (1 - df_clean["descuento"])
    df_clean["ticket_unitario"] = df_clean["importe"] / df_clean["unidades"]

display(df_clean.head())

## Agrupaciones y visualización

In [ ]:
resumen = df_clean.groupby(["region","canal"], as_index=False).agg(
    ventas_totales=("importe","sum"),
    clientes=("cliente_id","nunique"),
    ticket_medio=("importe","mean")
).sort_values("ventas_totales", ascending=False)
display(resumen.head(10))

ventas_dia = df_clean.groupby("fecha")["importe"].sum()
if HAS_MATPLOTLIB:
    plt.figure(figsize=(10,4))
    plt.plot(ventas_dia.index, ventas_dia.values)
    plt.title("Ventas diarias")
    plt.xlabel("Fecha")
    plt.ylabel("Importe")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib no disponible")
    display(ventas_dia.head(10))

## Mini-reto

In [ ]:
# Reto: identifica el canal con mayor importe total y argumenta qué región genera más ventas.
display(df_clean.groupby("canal").agg(
    importe_total=("importe","sum"),
    ticket_medio=("importe","mean"),
    operaciones=("cliente_id","count")
).sort_values("importe_total", ascending=False))

## Actividad entregable
1. Modifica el dataset o hiperparámetros.
2. Añade una breve interpretación de resultados.
3. Guarda el notebook ejecutado y exporta una versión HTML/PDF si se solicita.